In [1]:
import pandas as pd
import networkx as nx
from networkx.algorithms import community
import os

# Get project root (ULSE directory)
# The notebook is in the copenhagen/ directory, so the parent directory is the project root
current_dir = os.getcwd()
if os.path.basename(current_dir) == "copenhagen":
    # If the notebook is executed in the copenhagen/ directory
    project_root = os.path.dirname(current_dir)
    data_dir = os.path.join(current_dir, "original_data")
else:
    # If executed from the project root
    project_root = current_dir
    data_dir = os.path.join(project_root, "copenhagen", "original_data")

# Create output directory (relative path from project root)
output_dir = os.path.join(project_root, "data", "copenhagen")
os.makedirs(output_dir, exist_ok=True)

# Select data source ('bt_symmetric', 'calls', or 'sms')
# Using bt_symmetric for this run
selected_source = 'bt_symmetric'

print(f"Project root: {project_root}")
print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")
print(f"Selected data source: {selected_source}")


Project root: /home/matsumoto/ULSE
Data directory: /home/matsumoto/ULSE/copenhagen/original_data
Output directory: /home/matsumoto/ULSE/data/copenhagen
Selected data source: bt_symmetric


In [2]:
# Task 1: Create graph from fb_friends.csv and perform module detection for labeling
print("Task 1: Performing module detection on fb_friends.csv...")

# Load fb_friends.csv (header row is automatically recognized)
fb_friends_path = os.path.join(data_dir, "fb_friends.csv")
df_fb = pd.read_csv(fb_friends_path, comment='#')
# Convert node IDs to numeric type (to match with bt_symmetric, etc.)
df_fb['user_a'] = pd.to_numeric(df_fb['user_a'], errors='coerce')
df_fb['user_b'] = pd.to_numeric(df_fb['user_b'], errors='coerce')
# Remove NaN values
df_fb = df_fb.dropna(subset=['user_a', 'user_b'])

# Create graph
G = nx.Graph()
for _, row in df_fb.iterrows():
    G.add_edge(int(row['user_a']), int(row['user_b']))

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

# Community detection
communities_generator = community.greedy_modularity_communities(G)
communities = {}
for i, comm in enumerate(communities_generator):
    for node in comm:
        communities[node] = i
print(f"Number of detected communities: {len(communities_generator)}")

# Create node-to-label mapping
node_to_label = communities
print(f"Number of labeled nodes: {len(node_to_label)}")


Task 1: Performing module detection on fb_friends.csv...
Number of nodes: 800
Number of edges: 6429
Number of detected communities: 7
Number of labeled nodes: 800


In [3]:
# Task 2: Convert time data from seconds to days and hours
print(f"\nTask 2: Converting time data from seconds to days and hours... (data source: {selected_source})")

def convert_seconds_to_days(seconds):
    """Convert seconds to days (1 day = 86400 seconds)"""
    return int(float(seconds) / 86400.0)

def convert_seconds_to_hours(seconds):
    """Convert seconds to hours (1 hour = 3600 seconds)"""
    return int(float(seconds) / 3600.0)

# Configuration for each data source
source_config = {
    'bt_symmetric': {
        'filename': 'bt_symmetric.csv',
        'source_col': 'user_a',
        'target_col': 'user_b',
        'read_kwargs': {'comment': '#'},  # Header row is automatically recognized, comment lines are ignored
        # According to README, user_b = -1 (Empty scans) and user_b = -2 (external devices) should be excluded
        # The condition user_b >= 0 excludes -1 and -2
        'filter_func': lambda df: df[(df['user_a'] >= 0) & (df['user_b'] >= 0)]
    },
    'calls': {
        'filename': 'calls.csv',
        'source_col': 'caller',
        'target_col': 'callee',
        'read_kwargs': {'comment': '#'},  # Header row is automatically recognized, comment lines are ignored
        'filter_func': lambda df: df.dropna(subset=['caller', 'callee', 'timestamp'])
    },
    'sms': {
        'filename': 'sms.csv',
        'source_col': 'sender',
        'target_col': 'recipient',
        'read_kwargs': {'comment': '#'},  # Header row is automatically recognized, comment lines are ignored
        'filter_func': lambda df: df.dropna(subset=['sender', 'recipient', 'timestamp'])
    }
}

# Get configuration for selected data source
if selected_source not in source_config:
    raise ValueError(f"Unknown data source: {selected_source}. Please select one of 'bt_symmetric', 'calls', 'sms'.")

config = source_config[selected_source]
print(f"Processing {config['filename']}...")

# Load CSV file
file_path = os.path.join(data_dir, config['filename'])
read_kwargs = {**config['read_kwargs'], 'low_memory': False}
df = pd.read_csv(file_path, **read_kwargs)

# Convert numeric columns to numeric type
source_col = config['source_col']
target_col = config['target_col']
df[source_col] = pd.to_numeric(df[source_col], errors='coerce')
df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')

# Filtering
# For bt_symmetric: According to README, exclude user_b = -1 (Empty scans) and user_b = -2 (external devices)
# For other data sources: Exclude NaN values
df = config['filter_func'](df)

# Convert timestamp to days and hours
df['timestamp_days'] = df['timestamp'].apply(convert_seconds_to_days)
df['timestamp_hours'] = df['timestamp'].apply(convert_seconds_to_hours)

# Create edge lists for both daily and hourly data
selected_edges_daily = []
selected_edges_hourly = []
for _, row in df.iterrows():
    source = int(row[source_col])
    target = int(row[target_col])
    selected_edges_daily.append((source, target, row['timestamp_days']))
    selected_edges_hourly.append((source, target, row['timestamp_hours']))

print(f"  Added {len(selected_edges_daily)} edges for daily data")
print(f"  Added {len(selected_edges_hourly)} edges for hourly data")

# Sort edges chronologically (deterministic sorting by timestamp, then source, then target)
selected_edges_daily.sort(key=lambda x: (x[2], x[0], x[1]))  # Sort by timestamp, source, target
selected_edges_hourly.sort(key=lambda x: (x[2], x[0], x[1]))  # Sort by timestamp, source, target
print(f"\nSorted edges chronologically: {len(selected_edges_daily)} daily edges, {len(selected_edges_hourly)} hourly edges")



Task 2: Converting time data from seconds to days and hours... (data source: bt_symmetric)
Processing bt_symmetric.csv...
  Added 2426279 edges for daily data
  Added 2426279 edges for hourly data

Sorted edges chronologically: 2426279 daily edges, 2426279 hourly edges


In [4]:
# Task 3: Create output files (remap node IDs to consecutive numbers)
print(f"\nTask 3: Creating output files... (data source: {selected_source})")

# Create output directories for each data source and time granularity (shorten 'bt_symmetric' to 'bt')
source_suffix = 'bt' if selected_source == 'bt_symmetric' else selected_source
output_subdir_daily = os.path.join(project_root, "data", f"copenhagen_{source_suffix}_daily")
output_subdir_hourly = os.path.join(project_root, "data", f"copenhagen_{source_suffix}_hourly")
os.makedirs(output_subdir_daily, exist_ok=True)
os.makedirs(output_subdir_hourly, exist_ok=True)

# Collect all nodes (from selected data source - use daily edges for node collection)
all_nodes = set()
for source, target, _ in selected_edges_daily:
    all_nodes.add(source)
    all_nodes.add(target)

# Remap node IDs to consecutive numbers (mapping from smallest to largest)
sorted_nodes = sorted(all_nodes)
node_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted_nodes)}
print(f"Remapped node IDs: {len(sorted_nodes)} nodes (min: {min(sorted_nodes)}, max: {max(sorted_nodes)} -> 0-{len(sorted_nodes)-1})")

# Process daily data
print(f"\nProcessing daily data...")
remapped_edges_daily = []
for source, target, timestamp in selected_edges_daily:
    remapped_edges_daily.append((node_mapping[source], node_mapping[target], timestamp))

# Remove duplicate edges on the same day (same timestamp)
# Each edge appears only once per day
unique_edges_daily = set(remapped_edges_daily)
print(f"Daily data - Before deduplication: {len(remapped_edges_daily)} edges")
print(f"Daily data - After deduplication: {len(unique_edges_daily)} edges (reduced: {len(remapped_edges_daily) - len(unique_edges_daily)} edges)")

# Sort chronologically (by timestamp, then source, then target)
unique_edges_daily_sorted = sorted(unique_edges_daily, key=lambda x: (x[2], x[0], x[1]))

# Create copenhagen_<source>_daily.txt in format <source> <target> <timestamp>
copenhagen_daily_txt_path = os.path.join(output_subdir_daily, f"copenhagen_{source_suffix}_daily.txt")
with open(copenhagen_daily_txt_path, 'w') as f:
    for source, target, timestamp in unique_edges_daily_sorted:
        f.write(f"{source} {target} {timestamp}\n")
print(f"Created copenhagen_{source_suffix}_daily.txt: {len(unique_edges_daily_sorted)} edges")

# Process hourly data
print(f"\nProcessing hourly data...")
remapped_edges_hourly = []
for source, target, timestamp in selected_edges_hourly:
    remapped_edges_hourly.append((node_mapping[source], node_mapping[target], timestamp))

# Remove duplicate edges in the same hour (same timestamp)
# Each edge appears only once per hour
unique_edges_hourly = set(remapped_edges_hourly)
print(f"Hourly data - Before deduplication: {len(remapped_edges_hourly)} edges")
print(f"Hourly data - After deduplication: {len(unique_edges_hourly)} edges (reduced: {len(remapped_edges_hourly) - len(unique_edges_hourly)} edges)")

# Sort chronologically (by timestamp, then source, then target)
unique_edges_hourly_sorted = sorted(unique_edges_hourly, key=lambda x: (x[2], x[0], x[1]))

# Create copenhagen_<source>_hourly.txt in format <source> <target> <timestamp>
copenhagen_hourly_txt_path = os.path.join(output_subdir_hourly, f"copenhagen_{source_suffix}_hourly.txt")
with open(copenhagen_hourly_txt_path, 'w') as f:
    for source, target, timestamp in unique_edges_hourly_sorted:
        f.write(f"{source} {target} {timestamp}\n")
print(f"Created copenhagen_{source_suffix}_hourly.txt: {len(unique_edges_hourly_sorted)} edges")

# Create node2label.txt in format <node> <label>
# Use community labels detected from fb_friends
# Nodes in temporal data but not in fb_friends are assigned default label 0
# Create node2label.txt in both daily and hourly directories
node2label_daily_path = os.path.join(output_subdir_daily, "node2label.txt")
node2label_hourly_path = os.path.join(output_subdir_hourly, "node2label.txt")

# Assign labels (using remapped node IDs)
for node2label_path in [node2label_daily_path, node2label_hourly_path]:
    with open(node2label_path, 'w') as f:
        for old_node in sorted_nodes:
            new_node = node_mapping[old_node]
            label = node_to_label.get(old_node, 0)  # Nodes not in fb_friends get label 0
            f.write(f"{new_node} {label}\n")

print(f"\nCreated node2label.txt: {len(sorted_nodes)} nodes")
print(f"  Number of labels: {len(set(node_to_label.values()))}")

print(f"\nComplete! Output directories:")
print(f"  Daily: {output_subdir_daily}")
print(f"  Hourly: {output_subdir_hourly}")



Task 3: Creating output files... (data source: bt_symmetric)
Remapped node IDs: 692 nodes (min: 0, max: 845 -> 0-691)

Processing daily data...
Daily data - Before deduplication: 2426279 edges
Daily data - After deduplication: 188971 edges (reduced: 2237308 edges)
Created copenhagen_bt_daily.txt: 188971 edges

Processing hourly data...
Hourly data - Before deduplication: 2426279 edges
Hourly data - After deduplication: 455796 edges (reduced: 1970483 edges)
Created copenhagen_bt_hourly.txt: 455796 edges

Created node2label.txt: 692 nodes
  Number of labels: 7

Complete! Output directories:
  Daily: /home/matsumoto/ULSE/data/copenhagen_bt_daily
  Hourly: /home/matsumoto/ULSE/data/copenhagen_bt_hourly
